# MLflow Example: Tracking Two Model Training Runs

In [ ]:
! pip3 install sklearn

In [2]:
# === Model Training and Experimentation ===

# Import necessary libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits


In [3]:

# Load dataset
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

# Define a function to train a model and return its accuracy
def train_model(n_estimators, max_depth):
    # Train a RandomForestClassifier
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)

    # Make predictions
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    print(f"Model with n_estimators={n_estimators}, max_depth={max_depth} achieved accuracy={accuracy:.4f}")
    return model, accuracy

In [4]:

# Train models with different hyperparameter configurations
print("Training Model 1...")
model_1, acc_1 = train_model(n_estimators=20, max_depth=5)

Training Model 1...
Model with n_estimators=20, max_depth=5 achieved accuracy=0.9389


In [5]:
print("\nTraining Model 2...")
model_2, acc_2 = train_model(n_estimators=100, max_depth=10)



Training Model 2...
Model with n_estimators=100, max_depth=10 achieved accuracy=0.9722


Now adding MLFlow

In [ ]:
! pip3 install mlflow

In [6]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits

In [ ]:
!pip3 install dagshub

In [7]:
# get this code from dagshub
import dagshub
dagshub.init(repo_owner='DorDNS', repo_name='mlflow-with-daghub', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

/Users/doryandenis/Desktop/EFREI/Semestre 9/ML in 
Prod/mlflow-with-daghub/.venv/lib/python3.9/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=2d897bc5-f200-4d9d-8bb5-72c4402cdea1&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e4189b6f51bc8ec3999c9ef9730c675b7756810c8dcb74ef72b6e9fdfcd0902b




Accessing as DorDNS

Initialized MLflow to track repo "DorDNS/mlflow-with-daghub"

Repository DorDNS/mlflow-with-daghub initialized!

In [8]:
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

In [12]:
# Set MLflow experiment name. This will get created if it doesn't exist
experiment_name = "RandomForest_V1.0.1"
mlflow.set_experiment(experiment_name)


2026/11/16 16:06:42 INFO mlflow.tracking.fluent: Experiment with name 'RandomForest_V1.0.1' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/b890deae0f7d4746b1026afbe218db1d', creation_time=1794841602757, experiment_id='0', last_update_time=1794841602757, lifecycle_stage='active', name='RandomForest_V1.0.1', tags={}>

In [13]:
# Define a function to train a model, log parameters and metrics to MLflow
def train_and_log_model(n_estimators, max_depth, run_name):
    with mlflow.start_run(run_name=run_name): # <--
        # Train a RandomForestClassifier
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Make predictions
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)

        # Log parameters, metrics, and the model itself
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_metric("accuracy", accuracy)
        mlflow.sklearn.log_model(model, name="model", registered_model_name="digits_random_forest")

        print(f"Logged RandomForest model with n_estimators={n_estimators}, max_depth={max_depth}, accuracy={accuracy:.4f}")


In [14]:
# Train and log Model 1
print("Training and Logging Model 1...")
train_and_log_model(n_estimators=20, max_depth=5, run_name="random_forest_model_run_1")

Training and Logging Model 1...


2026/11/16 16:08:00 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'digits_random_forest'.
2026/11/16 16:08:05 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: digits_random_forest, version 1
Created version '1' of model 'digits_random_forest'.


Logged RandomForest model with n_estimators=20, max_depth=5, accuracy=0.9389
🏃 View run random_forest_model_run_1 at: https://dagshub.com/DorDNS/mlflow-with-daghub.mlflow/#/experiments/0/runs/211025e6c4704acb82faba417f3bcea1
🧪 View experiment at: https://dagshub.com/DorDNS/mlflow-with-daghub.mlflow/#/experiments/0


In [15]:
# Train and log Model 2
print("\nTraining and Logging Model 2...")
train_and_log_model(n_estimators=100, max_depth=10, run_name="random_forest_model_run_2")


Training and Logging Model 2...


2026/11/16 16:08:20 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'digits_random_forest' already exists. Creating a new version of this model...
2026/11/16 16:08:23 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: digits_random_forest, version 2
Created version '2' of model 'digits_random_forest'.


Logged RandomForest model with n_estimators=100, max_depth=10, accuracy=0.9722
🏃 View run random_forest_model_run_2 at: https://dagshub.com/DorDNS/mlflow-with-daghub.mlflow/#/experiments/0/runs/508ed36bf81343e5bb0bef20f41af1e5
🧪 View experiment at: https://dagshub.com/DorDNS/mlflow-with-daghub.mlflow/#/experiments/0


In [ ]:
# Instructions to visualize results
print("\nTo view the results, run the following command in your terminal:")
print("mlflow ui")
print("Then navigate to http://127.0.0.1:5000 to explore the experiment results.")